# Grafos

Un grafo dirigido es un par G = (V, E) donde V es el conjunto de nodos y E es el conjunto de aristas de la forma (u, v) donde u, v ∈ V.

\* En un grafo dirigido, puede haber una arista que vaya al mismo nodo.

Un grafo no dirigido es un par G = (V, E) donde V es el conjunto de nodos y E es el conjunto de aristas de la forma {u, v} donde u, v ∈ V, y u ≠ v.

##### Representación de grafos
- Lista de adyacencia: la celda i de la lista tiene una lista con los vecinos j tales que (i, j) ∈ E.
- Matriz de adyacencia: la celda [i][j] indica si la arista (i, j) ∈ E.
\* La complejidad en grafos puede depender de qué representación se usa.


Para detectar ciclos, asignaremos colores a los nodos:
- Negro: visitado no posterior (terminado), no hay ningún camino de ahí en adelante.
- Gris: visitado posterior, lo visitamos, pero aún hay camino (ciclo).
- Blanco: no visitado.

Este algoritmo determina un ciclo a partir de X.
```plain
cycleAfter(G, X):
    if X.color = gris : return true
    if X.color = negro : return false
    X.color = gris
    for Y tal que X -> Y:
        if cycleAfter(G, Y): return true
    X.color = negro
    return false
```

Pueden haber nodos que no sean alcanzables desde X.
```plain
isCyclic(G):
    for X in V(G):
        if X.color != blanco:
            continue
        if cycleAfter(G, X):
            return true
    return false
```
\* Este algoritmo avanza por aristas hasta más no poder (DFS, todas las rutas se exploran en profundidad). Complejidad O(V + E) (lineal en G).


## DFS
Depth First Search

\* Si recorremos un ABB con DFS, recorre todos los nodos en orden.

```plain
DFS(G):
    for u in V(G):
        u.color = blanco
    for u in V(G):
        if u.color == blanco:
            DFSVisit(G, u)
```
```plain
DFSVisit(G, u):
    u.color = gris
    for v in adyacentes(G, u):
        if v.color == blanco:
            DFSVisit(G, v)
    u.color = negro
```

Se pueden usar tiempos de descubrimiento y cierre para usar la noción de colores, así sabemos que si el intervalo de un nodo está contenido en el intervalo de otro, sabemos que viene después.

```plain
DFS(G):
    t = 1
    for u in V(G):
        u.start = 0
        u.end = 0
    for u in V(G):
        if u.start == 0:
            t = DFSVisit(G, u, t)
```
```plain
DFSVisit(G, u, t):
    u.start = t
    t = t + 1
    for v in adyacentes(G, u):
        if v.start == 0:
            DFSVisit(G, v, t)
    u.end = t
    t = t + 1
    return t
```
\* Para darnos cuenta de un nodo "gris", es cuando tiene start != 0, pero end == 0.

\* Los nodos negros son los que tienen start != 0 y end != 0.

\* El recorrido DFS puede generar un bosque de árboles independientes.

Dado G y u, v en V(G), diremos que u es descendiente de v al ejecutar DFS(G) si u es visitado por primera vez luego de v y antes de que v sea terminado. En tal caso diremos que ambos pertenecen al mismo árbol DFS. (u va después de v)

Dado G y u, v en V(G), luego de ejecutar DSF(G) la arista (u, v) puede ser:
- Arista de árbol: v fue descubierto al transitar (u, v).
- Arista hacia atrás: v fue descubierto antes que u, y está en el mismo árbol DFS (u es descendiente de v).
- Arista hacia adelante: si (u, v) no es árbol y v es descendiente de u.
- Arista cruzada: EOC.

\* Un grafo G es acíclico SSI DFS(G) no produce aristas hacia atrás.

#### Orden topológico

Es una secuencia lineal de los nodos de un grafo dirigido **acíclico** tal que para cada arista (u, v) ∈ E, u aparece antes que v en la secuencia (el orden de los nodos, en orden).

\* Si el árbol tiene un ciclo, no existe orden topológico.

```plain
TopSort(G):
    L = []
    DSP(G)
    Insertar en L nodos en orden decreciente según .end
    return L
```

Sea G un grafo dirigido, una componente fuertemente conectada (CFC) es un conjunto  maximal de nodos C ⊆ V(G) tal dados u, v ∈ C, existe un camino dirigido desde u hasta v.
\* Son como las clases de estados de markov (estocásticos).


### Algoritmo de Kosaraju
```plain
Kosaraju(G):
    DFS(G) // obtener tiempos de cierre
    G' = Transponer(G)
    DFS(G') // recorrer en orden decreciente de tiempos de cierre
```
\* La transposición de un grafo dirigido G = (V, E) es G' = (V, E') donde E' = {(v, u) | (u, v) ∈ E}.

Dado un grafo G dirigido, sean C1, ..., Ck sus CFC. Se define el grafo de componentes G^CFC según:
- V(G^CFC) = {C1, ..., Ck}
- Si (u, v) ∈ E(G) con u ∈ Ci y v ∈ Cj y i ≠ j, entonces (Ci, Cj) ∈ E(G^CFC).

\* El grafo de componentes es acíclico.
\* El grafo de componentes tiene un orden topológico.

_\* Es el grafo condensado._

### MST
Dado un grafo no dirigido G, un subgrafo T ⊆ G se dice un árbol de cobertura mínimo (MST) si:
1. T es un árbol.
2. V(T) = V(G).
3. No existe otro MST T' para G con menor costo total.

T es MST de G si:
1. No tiene ciclos.
2. Es cuna cobertura de todos los nodos de G.
3. Tiene costo mínimo.

Llamamos corte a una partición (V1, V2) de V(G) tal que V1 ∪ V2 = V(G) y V1 ∩ V2 = ∅, con V1, V2 ≠ ∅.

Diremos que una arista cruza el corte si uno de sus extremos está en V1 y el otro en V2.

##### Algoritmo de Kruskal
La idea es crear un bosque que va convergiendo en un único árbol.

Para un grafo G = (V, E), iteramos sobre las aristas _e_ en orden decreciente de costo:
- Si _e_ genera un ciclo al agregarla a T, la ignoramos.
- Si no genera un ciclo, la agregamos a T.

```plain
Kruskal(G):
    E = E ordenada por costo, de menor a mayor
    for e in E:
        if Agregar e a T no forma ciclo:
            T = T ∪ {e}
    return T
```

Dada una arista (u, v) podemos considerar los conjuntos:
- Nodos conectados con _u_ en T:
    V_u = {w | w está conectado con u con aristas de T}
- Nodos conectados con _v_ en T:
    V_v = {w | w está conectado con v con aristas de T}

La arista forma un ciclo en T SSI V_u = V_v.
\* Los árboles del bosque T forman conjuntos disjuntos.

Los conjuntos disjuntos se pueden manejar con una estructura de datos llamada _Union-Find_.

Se definen los representantes de cada conjunto disjunto. Luego se definen:
- Find(A): entrega el conjunto al que pertenece A (representante).
- Union(A, B): une los conjuntos a los que pertenecen A y B (el representante de un conjunto apuntará al representante del otro).

Se pueden almacenar los conjuntos disjuntos como arreglos de referencias, donde A[k] es el padre de k. Si k es representante, A[k] = k.

\* Tiene complejidad casi constante, O(α(n)), prácticamente O(1).

```plain
Kruskal(G):
    E = E ordenada por costo, de menor a mayor
    for v in V:
        MakeSet(v) // cada nodo es su propio conjunto
    T = ∅
    for e = (u, v) in E:
        if Find(u) != Find(v): // no forman ciclo
            T = T ∪ {e}
            Union(u, v)
    return T
```

\* Complejidad O(E log(V)) por el ordenamiento de las aristas.

##### Algoritmo de Prim
La idea es utilizar aristas que cruzan cortes para guiar la construcción del MST.

Prim construye un MST de forma codiciosa, partiendo de un nodo inicial y agregando aristas de menor costo que conectan nodos ya incluidos en el MST con nodos fuera del MST.

1. Iniciar con un nodo `s` (agregar a conjunto R).
2. Repetir hasta cubrir todos los nodos:
   1. Encontrar la arista de menor costo que cruza el corte (R, R').
   2. Agregar esa arista al MST.
   3. Mover el nodo destino de R' a R.

Para un grafo G = (V, E) y un nodo inicial v:
1. Sean R = {_v_} y R' = V - R
2. Sea _e_ la arista de menor costo que cruza de R a R' (corte).
3. Sea _u_ el nodo de _e_ que pertenece a R'.
4. Agregar _e_ al MST. Eliminar _u_ de R' y agregarlo a R.
5. Si quedan elementos en R', volver al paso 2.

\* Q usa como prioridad el valor d[v].

\* DecreaseKey(Q, v, d[v]) cambia la prioridad del elemento v.

Complejidad O(E log V).

# Heap
Una cola de prioridades es una EDD que permite:
- Almacenar datos según cierta prioridad.
- Consultar cuál es el dato más prioritario.
- Recorrer los datos en orden de prioridad.

\* No tengo acceso a todos los datos, sólo al más prioritario.

### Colas FIFO
First In First Out
_El primero en llegar es el primero en salir_ (Queue).

Es una cola donde:
- **Primer** elemento es el **más** prioritario, lleva más tiempo en la cola.
- **Último** elemento es el **menos** prioritario, lleva menos tiempo en la cola.

Operaciones:
* Inserción: se inserta al **final** de la cola.
  * Arreglo: O(1) (a menos que se llene).
  * Lista: O(1).
* Extracción: se elimina la **cabeza** de la cola.
  * Arreglo: O(1) si no reubicamos (mover al puntero, no a los elementos).
  * Lista: O(1).

### Colas LIFO
Last In First Out
_El último en llegar es el primero en salir_ (Stack).

Es una cola donde:
- **Primer** elemento es el **menos** prioritario, lleva más tiempo en la cola.
- **Último** elemento es el **más** prioritario, lleva menos tiempo en la cola.

Operaciones:
* Inserción: se inserta al **final** de la cola.
  * Arreglo: O(1) (recorriendo al revés).
  * Lista: O(1).
* Extracción: se elimina la **cabeza** de la cola.
  * Arreglo: O(1) si no reubicamos (mover al puntero, no a los elementos).
  * Lista: O(1).


##### Cola de prioridades
Una cola de prioridades o cola highest priority first out es una EDD que permite:
- Insertar un dato con una prioridad dada.
- Extraer el dato con mayor prioridad.
- Idealmente, **cambiar** la prioridad de un dato.
\* El orden de llegada no es equivalente a la posición en la cola.

Si la prioridad de una cola de prioridades A es el valor de los datos, y todos son naturales, tenemos 2 opciones:
- Arreglo sin orden:
  - Inserción al final: O(1).
  - Extracción del máximo: O(n) (recorrer todo el arreglo).
- Arreglo ordenado:
  - Inserción al final: O(n) (buscar posición e insertar).
  - Extracción del máximo: O(1).

##### Orden
- Total: todos los elementos de A están ordenados.
- Parcial: hay sub-sectores de A que están ordenados, y conocemos bien la división de los sub-sectores.

### Heap binario
Un Max heap binario H es un árbol binario tal que:
- H.left y H.right son Max heaps binarios.
- H.key > H.left.key
- H.key > H.right.key

\* Todo hijo tiene llaves menores que el padre, pero entre hermanos no hay restricción.

##### Balance
Los heaps se almacenan completándolos por nivel (como árboles binarios casi llenos). Lo que interesa es que antes de agregar un nivel, el último disponible debe estar completo.

\* No significa que para h niveles haya 2^h - 1 nodos.

Mantener los heaps balanceados permite minimizar la altura del árbol representado, e implementar el heap de forma compacta en un arreglo (no se necesitan punteros):
- El elemento H[k] es padre de H[2k + 1] y H[2k + 2].
- El padre del elemento H[k] es H[(k - 1) / 2].
- El primer elemento del nivel h es H[2^h - 1].
- Los 2^h elementos consecutivos corresponden al nivel h.

Al insertar y extraer:
1. Efectuar la operación manteniendo un árbol binario casi lleno.
2. Reestablecer la propiedad de heap.

##### Construcción de un heap

Se recorren los nodos del array en orden inverso (de abajo hacia arriba), y se aplica SiftDown para dejar el heap ordenado en cada iteración.

\* Complejidad O(n).

\* Esto es para un max heap, para un min heap es con SiftUp.

### Heapsort

Complejidad O(n) (BuildHeap) + O(n log n) (n extracciones de máximo) = O(n log n).

\* O(1) en memoria.

## BFS
Breadth First Search

Recorre un grafo por niveles, explorando todos los nodos vecinos antes de avanzar a los siguientes niveles.

\* Es especialmente útil para encontrar el camino más corto.

Podemos distinguir los nodos con colores:
- Blanco: no descubierto.
- Gris: descubierto con vecinos no descubiertos (pendientes).
- Negro: descubierto con vecinos descubiertos (terminado).

1. Recorre cada vecino del nodo
2. Para cada nodo, revisamos sus vecinos
   1. Cuando revisamos a todos los vecinos, volvemos al nivel anterior y repetimos con el siguiente nodo en la cola (tío).
3. Cuando se revisaron todos los caminos se marca el nodo como negro.
4. Se repite hasta que no queden nodos por explorar.

### Dijkstra
Es un algoritmo codicioso para problemas de rutas más cortas con subestructuras óptimas (costos no negativos).

1. Inicializa todos los nodos con color blanco, costo infinito y sin padre.
2. Establece el costo del nodo inicial a 0 y lo inserta en una cola de prioridades (min heap).
3. Mientras la cola de prioridades no esté vacía, extrae el nodo con el costo mínimo y actualiza los costos de sus vecinos si se encuentra un camino más corto.
4. Marca el nodo extraído como procesado (color negro) para evitar revisarlo nuevamente.

\* Encuentra las rutas más baratas desde s.

\* Puede haber más de una ruta con el mismo costo, Dijkstra encuentra una.

\* Tiempo O((V + E)log V) con un heap binario.

### Bellman-Ford

Este algoritmo usa una estrategia de programación dinámica:
1. Buscamos las rutas más cortas con a lo más _k_ aristas.
2. Podemos utilizar las rutas más cortas con a lo más _k-1_ aristas.

##### Subestructura óptima en rutas

Si el camino dirigido _u_ p=> x->y es una ruta más corta de _u_ a _y_, entonces _p_ es una ruta más corta de _u_ a _x_.

\* Un camino óptimo |p| <= _k_ no necesariamente contiene uno de largo |p'| = k-1. Siempre contiene uno de largo |p'| <= k-1.

\* Toda ruta más corta **no** tiene ciclos de costo negativo.


1. Setteamos el costo de **todos** los nodos como infinito (excepto el inicial)
2. Recorremos **todas** las aristar para cada iteración, pero actualizamos los costos sólo si el costo **no** es infinito, es decir, si ya recorrimos esa arista en una iteración anterior.
3. Cuando actualizamos, guardamos la arista de menor valor, y el padre del nodo (para reconstruir la ruta más barata).
4. Luego de actualizar los costos de todas las aristas hasta el nodo objetivo, tendremos guardada la ruta más barata por construcción.

\* Complejidad O(VE) en el peor caso.

\* Si el grafo tiene un ciclo de costo negativo, nunca podré encontrar un camino más corto, por lo que el camino más corto tiene a lo más V-1 aristas. Si luego de determinar todas las rutas más cortas con a lo más V-1 aristas, puedo seguir actualizando costos, entonces existe un ciclo de costo negativo.

\* La estrategia de programación dinámica no se ve en el código, está implícita en la forma de modelar la solución, luego el código es simplementa la implementación de la solución (pensada con programación dinámica).

_\* Diseñar primero antes de ponerse a codificar, primero pensar COMO resolver el problema, después hacerlo._


_\* Primero afilar el hacha, después cortar el árbol._

### Floyd-Warshall

Sea G, se define la matriz de adyacencia W de G con entradas:
- W[i][i] = 0
- W[i][j] = cost(i, j) si (i, j) ∈ E
- W[i][j] = ∞ si (i, j) ∉ E y i ≠ j

Definimos _P_ como el conjunto de caminos simples dirigidos en donde pasa por un nodo intermedio _k_.
```plain
_P_ := {c | c es el camino simple dirigido desde o hasta j y todo nodo intermedio _v_ ∈ {1, ..., k} ⊆ V}.
```
Sea _p_ el camino más barato de _P_.

\* Si _k_ no es nodo de _p_ (no está en la ruta más corta), se considera igualmente en la ruta "por el dominio" de _k_.

\* Si _k_ es nodo de _p_, se puede divitdir en 2 subcaminos, uno desde _i_ hasta _k_ (p_1) con intermedios en {1, ..., k} y otro desde _k_ hasta _j_ (p_2) con intermedios en {1, ..., k}.

Se define D^k[i][j] como el costo de ruta más corta de i hasta j tal que todos los nodos intermedios están en {1, ..., k}.

\* Con k = 0 no hay nodos intermedios posibles, esto obliga a considerar rutas sin intermedios (la más directa).

\* d^0[i][j] = W[i][j] (matriz de adyacencia).

\* Si no hay arista directa, d^0[i][j] = ∞.


Se define:
- Si _k_ >= 1: (se define el mejor entre el camino directo del paso anterior, y el camino que pasa por _k_ en el paso anterior).
```plain
D^k[i][j] = min( D^(k-1)[i][j], D^(k-1)[i][k] + D^(k-1)[k][j] )
```
- Si _k_ = 0:
```plain
D^k[i][j] = W[i][j]
```

\Complejida O(n^3).